# Supervised Fine-Tuning (SFT) for Agent Distillation

This notebook fine-tunes Qwen models on golden trajectories generated by DeepSeek-V3.

**Hardware Requirements:**
- GPU: A100 (40GB or 80GB recommended)
- Runtime: Set to GPU in Colab

**Models Supported:**
- Qwen/Qwen2.5-0.5B-Instruct
- Qwen/Qwen2.5-1.5B-Instruct
- Qwen/Qwen2.5-7B-Instruct

## 1. Setup and Installation

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install required packages for Unsloth and SFT
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install -q datasets wandb

In [ ]:
# Mount Google Drive to save models
from google.colab import drive
drive.mount('/content/drive')

## 2. Import Libraries

In [ ]:
import os
import torch
from unsloth import FastLanguageModel
import transformers.activations
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset
import json
from datetime import datetime

# Patch for 2026 Transformers compatibility
if not hasattr(transformers.activations, "PytorchGELUTanh"):
    transformers.activations.PytorchGELUTanh = transformers.activations.GELUTanh

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Number of GPUs: {torch.cuda.device_count()}")

## 3. Configuration

Adjust these parameters based on your needs:

In [ ]:
# ============================================
# CONFIGURATION PARAMETERS
# ============================================

# Model Selection: Choose one of the following
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # Options: 0.5B, 1.5B, 7B
# MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
# MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# Dataset Configuration
DATASET_FILE = "gsm8k_golden_trajectories_new.jsonl"  # Will be uploaded to Colab
# DATASET_FILE = "gsm8k_golden_trajectories.jsonl"  # Smaller dataset for testing

# Model & Tokenizer Settings
MAX_SEQ_LENGTH = 2048  # Max sequence length for training
LOAD_IN_4BIT = True  # Use 4-bit quantization to save memory

# LoRA Configuration
LORA_R = 16  # LoRA rank
LORA_ALPHA = 16  # LoRA alpha
LORA_DROPOUT = 0.0  # LoRA dropout (0 = no dropout)
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Training Hyperparameters (Optimized for A100)
PER_DEVICE_BATCH_SIZE = 8  # A100 can handle larger batch sizes
GRADIENT_ACCUMULATION_STEPS = 2  # Effective batch size = 8 * 2 = 16
LEARNING_RATE = 2e-4
MAX_STEPS = 500  # Adjust based on dataset size
WARMUP_STEPS = 50
LOGGING_STEPS = 10
SAVE_STEPS = 100  # Save checkpoints every N steps

# Optimization Settings
USE_FP16 = not torch.cuda.is_bf16_supported()  # Use FP16 if BF16 not available
USE_BF16 = torch.cuda.is_bf16_supported()  # A100 supports BF16 (better than FP16)
USE_GRADIENT_CHECKPOINTING = True  # Saves memory

# Output Configuration
OUTPUT_DIR = "./outputs_sft"
SAVE_TO_DRIVE = True  # Save final model to Google Drive
DRIVE_SAVE_PATH = f"/content/drive/MyDrive/agent_distillation/{MODEL_NAME.split('/')[-1]}_sft"

# Weights & Biases (Optional)
USE_WANDB = False  # Set to True if you want to log to W&B
WANDB_PROJECT = "agent-distillation-sft"

print(f"Configuration loaded successfully!")
print(f"Model: {MODEL_NAME}")
print(f"Dataset: {DATASET_FILE}")
print(f"Using BF16: {USE_BF16}, FP16: {USE_FP16}")
print(f"Effective Batch Size: {PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")

## 4. Upload Dataset

Upload your golden trajectories JSONL file here, or download from a URL:

In [ ]:
# Option 1: Upload file manually from your computer
from google.colab import files

print("Please upload your golden trajectories JSONL file:")
uploaded = files.upload()

# Get the uploaded filename
if uploaded:
    DATASET_FILE = list(uploaded.keys())[0]
    print(f"Dataset uploaded: {DATASET_FILE}")

In [ ]:
# Option 2: Download from Google Drive (if already uploaded)
# Uncomment and modify the path below if you've already uploaded to Drive

# import shutil
# drive_dataset_path = "/content/drive/MyDrive/agent_distillation/gsm8k_golden_trajectories_new.jsonl"
# DATASET_FILE = "gsm8k_golden_trajectories_new.jsonl"
# shutil.copy(drive_dataset_path, DATASET_FILE)
# print(f"Dataset copied from Drive: {DATASET_FILE}")

In [ ]:
# Verify dataset and show sample
with open(DATASET_FILE, 'r') as f:
    sample_data = [json.loads(line) for line in f.readlines()[:3]]

print(f"Total samples in dataset: {sum(1 for _ in open(DATASET_FILE))}")
print(f"\nSample trajectory:")
print(json.dumps(sample_data[0], indent=2))

## 5. Initialize Weights & Biases (Optional)

In [ ]:
if USE_WANDB:
    import wandb
    wandb.login()
    wandb.init(
        project=WANDB_PROJECT,
        name=f"{MODEL_NAME.split('/')[-1]}_sft_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
        config={
            "model_name": MODEL_NAME,
            "max_seq_length": MAX_SEQ_LENGTH,
            "lora_r": LORA_R,
            "learning_rate": LEARNING_RATE,
            "batch_size": PER_DEVICE_BATCH_SIZE,
            "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        }
    )
else:
    os.environ["WANDB_DISABLED"] = "true"

## 6. Load Model and Tokenizer

In [ ]:
print(f"Loading model: {MODEL_NAME}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,  # Auto-detect based on GPU (BF16 for A100)
)

print("Model loaded successfully!")
print(f"Model dtype: {model.dtype}")
print(f"Model device: {model.device}")

## 7. Add LoRA Adapters

In [ ]:
print("Adding LoRA adapters...")

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Use Unsloth's optimized gradient checkpointing
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

print("LoRA adapters added successfully!")

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"Total parameters: {total_params:,}")

## 8. Load and Preprocess Dataset

In [ ]:
print(f"Loading dataset from {DATASET_FILE}...")

# Load dataset
dataset = load_dataset("json", data_files=DATASET_FILE, split="train")

print(f"Dataset loaded with {len(dataset)} samples")
print(f"\nDataset columns: {dataset.column_names}")

# Show a sample
print(f"\nSample entry:")
print(dataset[0])

In [ ]:
# Format dataset for training
def formatting_prompts_func(examples):
    """
    Convert messages to chat template format.
    
    Expected format: {"messages": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}
    """
    instructions = examples["messages"]
    texts = []
    
    for messages in instructions:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)
    
    return {"text": texts}

# Apply formatting
print("Formatting dataset...")
dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=dataset.column_names,
    desc="Formatting prompts"
)

print(f"Dataset formatted successfully!")
print(f"\nFormatted sample (first 500 chars):")
print(dataset[0]["text"][:500] + "...")

In [ ]:
# Analyze sequence lengths
def get_token_lengths(examples):
    lengths = [len(tokenizer.encode(text)) for text in examples["text"]]
    return {"token_length": lengths}

dataset_with_lengths = dataset.map(
    get_token_lengths,
    batched=True,
    desc="Computing token lengths"
)

lengths = dataset_with_lengths["token_length"]
print(f"\nToken Length Statistics:")
print(f"  Min: {min(lengths)}")
print(f"  Max: {max(lengths)}")
print(f"  Mean: {sum(lengths)/len(lengths):.1f}")
print(f"  Samples exceeding {MAX_SEQ_LENGTH}: {sum(1 for l in lengths if l > MAX_SEQ_LENGTH)}")

## 9. Setup Training Arguments

In [ ]:
# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Configure training arguments optimized for A100
training_args = TrainingArguments(
    # Output
    output_dir=OUTPUT_DIR,
    
    # Training steps
    max_steps=MAX_STEPS,
    warmup_steps=WARMUP_STEPS,
    
    # Batch size and gradient accumulation
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    
    # Learning rate
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",  # Cosine annealing
    
    # Precision
    fp16=USE_FP16,
    bf16=USE_BF16,
    
    # Optimization
    optim="adamw_8bit",  # 8-bit AdamW saves memory
    weight_decay=0.01,
    max_grad_norm=1.0,
    
    # Logging
    logging_steps=LOGGING_STEPS,
    logging_dir=f"{OUTPUT_DIR}/logs",
    report_to="wandb" if USE_WANDB else "none",
    
    # Saving
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,  # Keep only last 3 checkpoints
    
    # Misc
    seed=42,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
)

print("Training arguments configured!")

## 10. Initialize Trainer

In [ ]:
print("Initializing SFT Trainer...")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=training_args,
    packing=False,  # Don't pack multiple samples into one sequence
)

print("Trainer initialized successfully!")
print(f"\nReady to train on {len(dataset)} samples")

## 11. Start Training

In [ ]:
print("="*60)
print("STARTING SUPERVISED FINE-TUNING")
print("="*60)
print(f"Model: {MODEL_NAME}")
print(f"Dataset samples: {len(dataset)}")
print(f"Max steps: {MAX_STEPS}")
print(f"Effective batch size: {PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Learning rate: {LEARNING_RATE}")
print("="*60)

# Start training
trainer.train()

print("\n" + "="*60)
print("TRAINING COMPLETED!")
print("="*60)

## 12. Save Model

In [ ]:
# Save LoRA adapters (lightweight)
print("Saving LoRA adapters...")
lora_save_path = f"{OUTPUT_DIR}/lora_adapters"
model.save_pretrained(lora_save_path)
tokenizer.save_pretrained(lora_save_path)
print(f"LoRA adapters saved to: {lora_save_path}")

In [ ]:
# Save merged model (full model with LoRA weights merged)
print("Saving merged model (this may take a few minutes)...")
merged_save_path = f"{OUTPUT_DIR}/merged_model"

model.save_pretrained_merged(
    merged_save_path,
    tokenizer,
    save_method="merged_16bit"  # Can also use "merged_4bit" to save space
)

print(f"Merged model saved to: {merged_save_path}")

In [ ]:
# Copy to Google Drive for persistent storage
if SAVE_TO_DRIVE:
    import shutil
    
    print(f"Copying model to Google Drive...")
    os.makedirs(os.path.dirname(DRIVE_SAVE_PATH), exist_ok=True)
    
    # Copy LoRA adapters
    lora_drive_path = f"{DRIVE_SAVE_PATH}_lora"
    if os.path.exists(lora_drive_path):
        shutil.rmtree(lora_drive_path)
    shutil.copytree(lora_save_path, lora_drive_path)
    print(f"LoRA adapters saved to Drive: {lora_drive_path}")
    
    # Copy merged model
    merged_drive_path = f"{DRIVE_SAVE_PATH}_merged"
    if os.path.exists(merged_drive_path):
        shutil.rmtree(merged_drive_path)
    shutil.copytree(merged_save_path, merged_drive_path)
    print(f"Merged model saved to Drive: {merged_drive_path}")
    
    print("\nAll models saved to Google Drive successfully!")

## 13. Test the Fine-Tuned Model

In [ ]:
# Enable inference mode
FastLanguageModel.for_inference(model)

# Test with a sample GSM8K problem
test_question = """A car travels 60 miles per hour for 2 hours, then 40 miles per hour for 1.5 hours. What is the total distance traveled?"""

messages = [
    {"role": "user", "content": test_question}
]

# Format with chat template
formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Tokenize
inputs = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")

# Generate
print("Generating response...\n")
outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
)

# Decode and print
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

print("="*60)
print("TEST GENERATION")
print("="*60)
print(f"Question: {test_question}\n")
print(f"Response:\n{response}")
print("="*60)

## 14. Upload Model to Hugging Face Hub (Optional)

In [ ]:
# Uncomment to push to Hugging Face Hub
# from huggingface_hub import login
# 
# login()  # You'll need to provide your HF token
# 
# hf_username = "your-username"  # Replace with your HF username
# model_repo_name = f"{MODEL_NAME.split('/')[-1]}-gsm8k-sft"
# 
# model.push_to_hub_merged(
#     f"{hf_username}/{model_repo_name}",
#     tokenizer,
#     save_method="merged_16bit",
#     private=True  # Set to False to make public
# )
# 
# print(f"Model uploaded to: https://huggingface.co/{hf_username}/{model_repo_name}")

## 15. Training Summary and Next Steps

In [ ]:
print("="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"Model: {MODEL_NAME}")
print(f"Dataset: {DATASET_FILE} ({len(dataset)} samples)")
print(f"Max steps: {MAX_STEPS}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Batch size: {PER_DEVICE_BATCH_SIZE} x {GRADIENT_ACCUMULATION_STEPS} = {PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"\nModel saved to:")
print(f"  - Local: {merged_save_path}")
if SAVE_TO_DRIVE:
    print(f"  - Google Drive: {DRIVE_SAVE_PATH}_merged")
print("="*60)

print("\n📝 Next Steps:")
print("1. Download the model from Google Drive or use it directly in evaluation")
print("2. Run evaluation on GSM8K test set using eval_sft.py")
print("3. Compare performance with baseline (Zero-Shot CoT)")
print("4. Proceed to GRPO optimization phase")
print("\n✅ SFT Phase Complete!")